# 📓 Semana 14 · Dia 6 — Entregável: agente Text-to-SQL seguro e auditável

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | GenAI Engineer Associate |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Agente final com 4+ ferramentas |

---


## 📖 Teoria — O agente que você construiu

```
pergunta → guardrail → agente (LLM + tools UC)
    → Text-to-SQL validado (SELECT only + auto-correção)
    → memória + auditoria (Delta)
    → avaliação (golden set + traces)
```


### 💻 Na prática — Agente integrado

Monte o agente completo em um notebook.


In [ ]:
# Tools finais do agente
def receita_por_pais(pais: str) -> str:
    """Receita total de um país (tabela Ouro)."""
    r = spark.sql(f"SELECT receita_total FROM workspace.ouro.receita_por_pais WHERE UPPER(Country) = UPPER('{pais}')").collect()
    return str(r[0][0]) if r else "País não encontrado"

def top_produtos(n: int = 5) -> str:
    """Top n produtos por receita."""
    rows = spark.sql(f"SELECT Description, receita_total FROM workspace.ouro.top_produtos ORDER BY receita_total DESC LIMIT {n}").collect()
    return "; ".join(f"{r[0]}: {r[1]}" for r in rows)

def vendas_por_periodo(de: str, ate: str) -> str:
    """Receita entre duas datas (YYYY-MM-DD)."""
    r = spark.sql(f"SELECT SUM(receita_total) FROM workspace.ouro.vendas_por_dia WHERE data_venda BETWEEN '{de}' AND '{ate}'").collect()
    return str(r[0][0] or 0)
print("4 ferramentas prontas (inclui UC function do dia 1).")

In [ ]:
# Montar agente com memória + auditoria
from langchain_community.chat_models import ChatDatabricks
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool
llm = ChatDatabricks(endpoint="databricks-llama-3-1-70b", temperature=0)
tools = [tool(receita_por_pais), tool(top_produtos), tool(vendas_por_periodo)]
prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é o assistente de dados do varejo. Use as ferramentas."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")])
agente_final = AgentExecutor(agent=create_tool_calling_agent(llm, tools, prompt),
                              tools=tools, verbose=True)
print("Agente final montado.")

In [ ]:
# Teste completo
p = "Qual a receita do United Kingdom?"
ok, msg = guardrail(p)
if ok:
    resp = agente_final.invoke({"input": p})
    registrar(p, resp["output"], "", True)
    print("Resposta:", resp["output"])
else:
    print("Bloqueado:", msg)

### 💻 Na prática — Validação final

1. Rode 5 perguntas variadas.
2. Confira a tabela de auditoria.
3. Rode a avaliação no golden set.
4. Documente o agente no README.


In [ ]:
# Auditoria final
display(spark.sql("SELECT * FROM workspace.audit.log_agente ORDER BY ts DESC LIMIT 10"))

> 🎯 **Dica de prova**: Entrevista: 'descreva um agente de dados seguro' → guardrails + tools governadas + SELECT-only + auditoria + avaliação contínua. Decore essa lista — ela vale vaga.


## 🎯 Exercícios de fixação

**1.** Rode o agente com 5 perguntas e registre tudo na auditoria.

**2.** O que falta para colocar esse agente em produção? (responda com os 7 pilares)


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Produção

Servir via Model Serving/Agent Framework (Semana 15), monitorar custo/latência, guardrails avançados e approvação humana para ações.

**2.** 7 pilares

Qualidade (avaliação), custo (gateway), latência (endpoint), segurança (guardrails), governança (UC), observabilidade (traces), escala (serving).



## ✅ Checklist de fechamento

- [ ] Construí agente com tools + UC functions.
- [ ] Text-to-SQL seguro (SELECT only + auto-correção).
- [ ] Memória, guardrails e auditoria implementados.
- [ ] Avaliei o agente com golden set + traces.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*